# AF2-SPDS refinement — static audit
Checks AF2CUE1 and AF2DECAY1 without training or test access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,shutil,subprocess,sys,time
from pathlib import Path
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/af2-signal-preservation-deep-supervision'; os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone,cwd=WORK)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
AF2_REL='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
PROJECT=resolve_drive_project_root(required_relative_paths=(AF2_REL,)); AF2=require_project_artifact(PROJECT,AF2_REL)
OUTPUT=PROJECT/'experiments/faruq-v3-af2-spds-refinement-v1'; OUTPUT.mkdir(parents=True,exist_ok=True); STATIC=OUTPUT/'static_audit.json'
print('AF2:',AF2); print('STATIC:',STATIC)

In [ ]:
from coffee_detector.af2_spds.refinement_audit import run_af2_spds_refinement_static_audit
audit=run_af2_spds_refinement_static_audit(AF2,STATIC,device='0')
print('ARMS:',audit['arms']); print('GATES:',audit['gates']); print('DECISION:',audit['decision']); print('SAVED:',STATIC)
assert audit['decision']=='PASS','STOP: jangan training.'
print('PASS: AF2CUE1 dan AF2DECAY1 boleh dijalankan paralel; test terkunci.')